# 03 · Separação treino/teste

Divide a base limpa em treino (80%) e teste (20%), com estratificação pela variável alvo, e salva os dois conjuntos em `data/processed/`.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.model_selection import train_test_split

from scroll_sono.config import (ID_COL, RANDOM_STATE, TARGET, TARGET_CLASSES, TEST_DATA_PATH,
                                TEST_SIZE, TRAIN_DATA_PATH)
from scroll_sono.data import load_clean, load_test, load_train, save_train_test

In [2]:
df = load_clean()
print(f"Base limpa: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

Base limpa: 8,500 linhas × 18 colunas


## 1. Separação

As classes são desbalanceadas (`Moderate Debt` tem 52,5% dos registros e `Severe Sleep Debt` só 7,6%), por isso a divisão é **estratificada** por `sleep_debt_category`: treino e teste ficam com as mesmas proporções de cada classe. O `random_state` fixo torna a divisão reproduzível.

A separação é única e vale para os dois cenários de atributos, já que todas as colunas são mantidas nos dois conjuntos.

In [3]:
train, test = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE
)
print(f"Treino: {len(train):,} linhas ({len(train) / len(df):.0%})")
print(f"Teste:  {len(test):,} linhas ({len(test) / len(df):.0%})")

Treino: 6,800 linhas (80%)
Teste:  1,700 linhas (20%)


## 2. Verificação das proporções das classes

In [4]:
proporcoes = pd.DataFrame({
    "base limpa (%)": df[TARGET].value_counts(normalize=True),
    "treino (%)": train[TARGET].value_counts(normalize=True),
    "teste (%)": test[TARGET].value_counts(normalize=True),
}).reindex(TARGET_CLASSES).mul(100)
proporcoes["diferença treino × teste (p.p.)"] = (proporcoes["treino (%)"] - proporcoes["teste (%)"]).abs()
proporcoes.round(2)

,base limpa (%),treino (%),teste (%),diferença treino × teste (p.p.)
sleep_debt_category,,,,
Optimal Recovery,16.32,16.32,16.29,0.03
Mild Deficit,23.58,23.57,23.59,0.01
Moderate Debt,52.49,52.49,52.53,0.04
Severe Sleep Debt,7.61,7.62,7.59,0.03


## 3. Verificação dos conjuntos

In [5]:
verificacoes = pd.Series({
    "linhas no treino + teste = base limpa": len(train) + len(test) == len(df),
    "registros em comum (user_id)": len(set(train[ID_COL]) & set(test[ID_COL])),
    "colunas iguais às da base limpa": list(train.columns) == list(test.columns) == list(df.columns),
    "todas as classes no teste": set(test[TARGET]) == set(TARGET_CLASSES),
})
verificacoes.to_frame("resultado")

,resultado
linhas no treino + teste = base limpa,True
registros em comum (user_id),0
colunas iguais às da base limpa,True
todas as classes no teste,True


## 4. Salvar os conjuntos

In [6]:
save_train_test(train, test)
for caminho in (TRAIN_DATA_PATH, TEST_DATA_PATH):
    print("Salvo em:", caminho.relative_to(caminho.parents[2]))

Salvo em: data/processed/train.csv
Salvo em: data/processed/test.csv


In [7]:
# Os conjuntos salvos são lidos de volta iguais aos que foram gerados?
pd.testing.assert_frame_equal(load_train(), train.reset_index(drop=True))
pd.testing.assert_frame_equal(load_test(), test.reset_index(drop=True))
print("Leitura conferida:", load_train().shape, load_test().shape)

Leitura conferida: (6800, 18) (1700, 18)


## Resumo

| Conjunto | Linhas | Arquivo |
|---|---|---|
| Treino | 6.800 (80%) | `data/processed/train.csv` |
| Teste | 1.700 (20%) | `data/processed/test.csv` |

As proporções das classes no treino e no teste diferem em no máximo 0,04 ponto percentual, e nenhum registro aparece nos dois conjuntos. O treino é a entrada da etapa 4 (análise exploratória); o teste só é usado na avaliação final.